In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
from datetime import datetime

CATALOG = "intelligent_etl"
SCHEMA  = "default"
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {RUN_ID}")

Run ID: 20260513_132005


In [0]:
bronze_df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")

typed_df = (
    bronze_df
    .withColumn("quantity",     F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price",   F.col("unit_price").cast(DoubleType()))
    .withColumn("total_amount", F.col("total_amount").cast(DoubleType()))
    .withColumn("discount_pct", F.col("discount_pct").cast(DoubleType()))
    .withColumn("order_date",
        F.to_timestamp(F.col("order_date"), "yyyy-MM-dd HH:mm:ss"))
)

print("Types after casting:")
typed_df.printSchema()

Types after casting:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- region: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _source: string (nullable = true)



In [0]:
window = Window.partitionBy("order_id").orderBy(F.col("_ingestion_ts").desc())

deduped_df = (
    typed_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

dupes_removed = typed_df.count() - deduped_df.count()
print(f"Rows before dedup: {typed_df.count():,}")
print(f"Rows after dedup:  {deduped_df.count():,}")
print(f"Duplicates removed: {dupes_removed:,}")

Rows before dedup: 10,000
Rows after dedup:  9,879
Duplicates removed: 121


In [0]:
VALID_STATUSES = ["Completed", "Shipped", "Processing", "Cancelled", "Returned"]
VALID_REGIONS  = ["North", "South", "East", "West", "Central"]

silver_df = (
    deduped_df

    # 1. Null in any required field
    .withColumn("dq_has_required_null",
        F.col("order_id").isNull()     |
        F.col("customer_id").isNull()  |
        F.col("product_id").isNull()   |
        F.col("total_amount").isNull() |
        F.col("order_date").isNull())

    # 2. Zero or negative amount
    .withColumn("dq_invalid_amount",
        (F.col("total_amount") <= 0) | (F.col("unit_price") <= 0))

    # 3. Future order date
    .withColumn("dq_future_date",
        F.col("order_date") > F.expr("current_timestamp() + interval 1 hour"))

    # 4. Invalid status
    .withColumn("dq_invalid_status",
        ~F.col("status").isin(VALID_STATUSES) & F.col("status").isNotNull())

    # 5. Invalid region
    .withColumn("dq_invalid_region",
        ~F.col("region").isin(VALID_REGIONS) & F.col("region").isNotNull())

    # 6. Quantity out of range
    .withColumn("dq_invalid_quantity",
        (F.col("quantity") < 1) | (F.col("quantity") > 999))

    # 7. Discount out of range
    .withColumn("dq_invalid_discount",
        (F.col("discount_pct") < 0) | (F.col("discount_pct") > 1))

    # 8. Amount mismatch
    .withColumn("dq_amount_mismatch",
        F.abs(
            F.col("total_amount") -
            (F.col("unit_price") * F.col("quantity") *
             (1 - F.coalesce(F.col("discount_pct"), F.lit(0))))
        ) > F.col("unit_price") * 0.01)
)

print("DQ flags added. Checking how many rows fail each check:")
dq_cols = [c for c in silver_df.columns if c.startswith("dq_")]
for col in dq_cols:
    count = silver_df.filter(F.col(col) == True).count()
    print(f"  {col}: {count:,} rows")

DQ flags added. Checking how many rows fail each check:
  dq_has_required_null: 102 rows
  dq_invalid_amount: 99 rows
  dq_future_date: 119 rows
  dq_invalid_status: 0 rows
  dq_invalid_region: 0 rows
  dq_invalid_quantity: 0 rows
  dq_invalid_discount: 0 rows
  dq_amount_mismatch: 9,632 rows


In [0]:
# Derived columns
silver_df = (
    silver_df
    .withColumn("order_hour",  F.hour("order_date"))
    .withColumn("order_dow",   F.dayofweek("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_year",  F.year("order_date"))
    .withColumn("revenue_tier",
        F.when(F.col("total_amount") >= 50000, "High")
         .when(F.col("total_amount") >= 10000, "Medium")
         .otherwise("Low"))
    .withColumn("_null_count",
        sum(F.col(c).isNull().cast("int") for c in [
            "customer_id","product_id","quantity","unit_price",
            "total_amount","order_date","region","payment_method","status"
        ]))
    .withColumn("_silver_ts", F.current_timestamp())
)

# Write Silver table
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")
)

print("Silver table written")
spark.table(f"{CATALOG}.{SCHEMA}.silver_orders").count()

Silver table written


9879

In [0]:
silver = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")

print(f"Total rows: {silver.count():,}")
print("\nDQ Flag Summary:")
print("="*45)

dq_cols = [c for c in silver.columns if c.startswith("dq_")]
for col in dq_cols:
    count = silver.filter(F.col(col) == True).count()
    pct   = count / silver.count() * 100
    print(f"  {col:<30} {count:>5,} rows  ({pct:.1f}%)")

print("\nSample flagged rows:")
silver.filter(F.col("dq_has_required_null") == True).show(3, truncate=True)

Total rows: 9,879

DQ Flag Summary:
  dq_has_required_null             102 rows  (1.0%)
  dq_invalid_amount                 99 rows  (1.0%)
  dq_future_date                   119 rows  (1.2%)
  dq_invalid_status                  0 rows  (0.0%)
  dq_invalid_region                  0 rows  (0.0%)
  dq_invalid_quantity                0 rows  (0.0%)
  dq_invalid_discount                0 rows  (0.0%)
  dq_amount_mismatch             9,632 rows  (97.5%)

Sample flagged rows:
+--------------------+-----------+----------+------------+-----------+--------+----------+------------+-------------------+-------+--------------+---------+------------+--------------------+---------------+--------------+--------------------+-----------------+--------------+-----------------+-----------------+-------------------+-------------------+------------------+----------+---------+-----------+----------+------------+-----------+--------------------+
|            order_id|customer_id|product_id|product_name|   cat

In [0]:
# Fix dq_amount_mismatch — remove the discount from the check
# Our data has total_amount = unit_price * quantity (discount not applied to total)
# So correct check is: total_amount ≈ unit_price * quantity

silver_fixed = (
    spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
    .withColumn("dq_amount_mismatch",
        F.abs(
            F.col("total_amount") -
            (F.col("unit_price") * F.col("quantity"))
        ) > F.col("unit_price") * 0.01
    )
)

# Overwrite Silver with fix
(
    silver_fixed
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")
)

# Recheck DQ summary
silver = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
print(f"Total rows: {silver.count():,}")
print("\nDQ Flag Summary (fixed):")
print("="*45)
dq_cols = [c for c in silver.columns if c.startswith("dq_")]
for col in dq_cols:
    count = silver.filter(F.col(col) == True).count()
    pct   = count / silver.count() * 100
    print(f"  {col:<30} {count:>5,} rows  ({pct:.1f}%)")

Total rows: 9,879

DQ Flag Summary (fixed):
  dq_has_required_null             102 rows  (1.0%)
  dq_invalid_amount                 99 rows  (1.0%)
  dq_future_date                   119 rows  (1.2%)
  dq_invalid_status                  0 rows  (0.0%)
  dq_invalid_region                  0 rows  (0.0%)
  dq_invalid_quantity                0 rows  (0.0%)
  dq_invalid_discount                0 rows  (0.0%)
  dq_amount_mismatch                 0 rows  (0.0%)
